# Anode: train the corrosion segmentation model (Stage 6a)

Trains a U-Net (segmentation_models_pytorch, MIT) with a ResNet-18 encoder to outline rust on
coated steel, then evaluates it on a **held-out test split** with the same harness the gold set
will use in Stage 6b.

Run in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). You need:

1. A free Roboflow account. Add its API key as a Colab secret named `ROBOFLOW_API_KEY` (key icon on the left).
2. Optional: a Hugging Face token as a Colab secret named `HF_TOKEN`, to upload the weights for the model server.

**Licences.** Only datasets under CC BY 4.0 or more permissive are used (Stage 6 decision). Before
adding a dataset below, open its Roboflow page, read its licence and copy it exactly. The notebook
refuses anything else and writes the attribution file.

In [ ]:
!pip -q install segmentation-models-pytorch==0.5.0 roboflow==1.1.50
!git clone -q https://github.com/Muddyscode/Anode.git
import sys

sys.path.insert(0, "Anode/model")

## 1. Datasets

Fill in each dataset after checking its licence on its page. The PRD's starting point is the
Roboflow Universe corrosion instance-segmentation set; confirm its licence before use.

In [ ]:
from anode_model.data import Dataset, attribution_markdown, check_licence  # noqa: F401 (Dataset is used in the list below)

DATASETS = [
    # Dataset(name="Corrosion instance segmentation", url="https://universe.roboflow.com/failure-sjoo6/corrosion-instance-segmentation-sfcpc-1yuxj",
    #         licence="<copy from the page>", workspace="failure-sjoo6", project="corrosion-instance-segmentation-sfcpc-1yuxj", version=1),
]
RUST_CATEGORIES = None  # e.g. {"corrosion", "rust"}; None treats every annotated class as rust

assert DATASETS, "Add at least one dataset above"
for ds in DATASETS:
    check_licence(ds)
open("ATTRIBUTION.md", "w").write(attribution_markdown(DATASETS))
print(open("ATTRIBUTION.md").read())

## 2. Download and build masks

In [ ]:
import json
from pathlib import Path

import numpy as np
from google.colab import userdata
from PIL import Image
from roboflow import Roboflow

from anode_model.data import coco_masks, split_of

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
DATA = Path("data")
for split in ("train", "val", "test"):
    (DATA / split / "images").mkdir(parents=True, exist_ok=True)
    (DATA / split / "masks").mkdir(parents=True, exist_ok=True)

seen = set()
for ds in DATASETS:
    loc = Path(rf.workspace(ds.workspace).project(ds.project).version(ds.version).download("coco-segmentation").location)
    for coco_path in loc.rglob("_annotations.coco.json"):
        masks = coco_masks(json.loads(coco_path.read_text()), RUST_CATEGORIES)
        for file_name, mask in masks.items():
            if file_name in seen:
                continue  # the same image in two datasets goes to one split only
            seen.add(file_name)
            split = split_of(file_name)  # our own split, so the test set is never trained on
            stem = Path(file_name).stem
            Image.open(coco_path.parent / file_name).convert("RGB").save(DATA / split / "images" / f"{stem}.jpg", quality=95)
            Image.fromarray((mask * 255).astype(np.uint8)).save(DATA / split / "masks" / f"{stem}.png")

for split in ("train", "val", "test"):
    print(split, len(list((DATA / split / "images").iterdir())))

## 3. Train

In [ ]:
import random
import time

import cv2
import segmentation_models_pytorch as smp
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as TorchDataset

SIZE, EPOCHS, BATCH, LR = 512, 30, 8, 3e-4
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)


class Rust(TorchDataset):
    def __init__(self, split, augment):
        self.images = sorted((DATA / split / "images").iterdir())
        self.split, self.augment = split, augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        img = np.asarray(Image.open(self.images[i]).convert("RGB"))
        mask = np.asarray(Image.open(DATA / self.split / "masks" / f"{self.images[i].stem}.png")) > 0
        img = cv2.resize(img, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask.astype(np.uint8), (SIZE, SIZE), interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if random.random() < 0.5:
                img, mask = img[:, ::-1], mask[:, ::-1]
            if random.random() < 0.5:
                img, mask = img[::-1], mask[::-1]
            img = np.clip(img * random.uniform(0.8, 1.2) + random.uniform(-20, 20), 0, 255)
        x = ((img.astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1)
        return torch.from_numpy(x.copy()), torch.from_numpy(mask.copy()[None].astype(np.float32))


train = DataLoader(Rust("train", True), batch_size=BATCH, shuffle=True, num_workers=2)
val = DataLoader(Rust("val", False), batch_size=BATCH, num_workers=2)
model = smp.Unet(encoder_name="resnet18", encoder_weights="imagenet", in_channels=3, classes=1).to(device)
dice, bce = smp.losses.DiceLoss("binary"), torch.nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)


def val_iou():
    model.eval()
    inter = union = 0
    with torch.no_grad():
        for x, y in val:
            p = (model(x.to(device)).sigmoid() >= 0.5).float()
            y = y.to(device)
            inter += (p * y).sum().item()
            union += ((p + y) > 0).sum().item()
    model.train()
    return inter / union if union else 1.0


best = -1
for epoch in range(EPOCHS):
    t = time.time()
    for x, y in train:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = dice(logits, y) + bce(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    sched.step()
    iou = val_iou()
    if iou > best:
        best = iou
        torch.save(model.state_dict(), "anode-unet.pt")
    print(f"epoch {epoch + 1}/{EPOCHS} loss {loss.item():.3f} val IoU {iou:.3f} best {best:.3f} ({time.time() - t:.0f}s)")

## 4. Evaluate on the held-out test set

Same harness, cut-offs and metrics the gold set will use in Stage 6b.

In [ ]:
from datetime import date

from anode_model.evaluate import evaluate, load_pairs, report_markdown
from anode_model.predictor import UNetPredictor

VERSION = f"{date.today().isoformat()}-unet-r18"
summary = evaluate(load_pairs(DATA / "test" / "images", DATA / "test" / "masks"), UNetPredictor(Path("anode-unet.pt"), VERSION))
report = report_markdown(summary, "Public held-out test set", VERSION)
open("evaluation-public.md", "w").write(report)
print(report)

## 5. Publish the weights

Downloads the weights, the report and the attribution. With `HF_TOKEN` set, also uploads them to a
Hugging Face model repository the model server loads from. Paste `evaluation-public.md` into
`docs/evaluation.md` and `ATTRIBUTION.md` into `model/ATTRIBUTION.md` in a pull request.

In [ ]:
from google.colab import files

for f in ("anode-unet.pt", "evaluation-public.md", "ATTRIBUTION.md"):
    files.download(f)

HF_REPO = None  # e.g. "your-hf-username/anode-unet"
if HF_REPO:
    from huggingface_hub import HfApi

    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HF_REPO, exist_ok=True, private=True)
    for f in ("anode-unet.pt", "evaluation-public.md", "ATTRIBUTION.md"):
        api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=HF_REPO, commit_message=f"Anode model {VERSION}")